# 05 — Qdrant Indexing & ANN Search

**CineMind Phase 1 · Step 5** — Load trained vectors into Qdrant and test
fast Approximate Nearest Neighbour (ANN) search.

### Why Qdrant?

Numpy search over 1,682 movies is instant, but the same brute-force search over
1 million movies would take seconds per request.  Qdrant uses an **HNSW** index
to find nearest vectors in milliseconds regardless of scale — it is the search
engine the live API will use.

**Two modes (auto-detected):**
- Qdrant on `localhost:6333` → uses the real database
- No server → falls back to numpy so you can follow along without Docker

To start Qdrant:
```bash
docker run -p 6333:6333 qdrant/qdrant
```

**Requires:** `artifacts/item_vecs.npy` (from notebook 04)

In [1]:
import os, sys

# Run from project root regardless of where Jupyter was launched
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

# Make src/ importable
if not any('src' in p for p in sys.path):
    sys.path.insert(0, 'src')

import cinemind_utils as cu
print("cwd:", os.getcwd())

import numpy as np
import pandas as pd
COLLECTION = "collab_vectors"

cwd: C:\Users\shaur\Desktop\My codes shaurya\Content Recommendation system\cinemind _phase1\cinemind_phase1


## 1 · Load vectors

In [2]:
item_vecs = np.load("artifacts/item_vecs.npy").astype(np.float32)
items     = pd.read_csv("data/items.csv")
title_of  = dict(zip(items.movie_id, items.title))
n, dim    = item_vecs.shape
print(f"Loaded {n:,} movie vectors of dimension {dim}.")

Loaded 1,683 movie vectors of dimension 64.


## 2 · Connect to Qdrant (or fall back to numpy)

In [3]:
use_qdrant = False
client     = None

try:
    from qdrant_client import QdrantClient
    from qdrant_client.models import Distance, VectorParams, PointStruct
    client = QdrantClient(host="localhost", port=6333, timeout=3)
    client.get_collections()   # ping
    use_qdrant = True
    print("Connected to Qdrant on localhost:6333")
except Exception as e:
    print(f"No Qdrant server ({type(e).__name__}). Using numpy fallback.")
    print("Start Qdrant with:  docker run -p 6333:6333 qdrant/qdrant")

Connected to Qdrant on localhost:6333


## 3 · Index the vectors

In [4]:
if use_qdrant:
    client.recreate_collection(
        collection_name=COLLECTION,
        vectors_config=VectorParams(size=dim, distance=Distance.COSINE),
    )
    points = [
        PointStruct(id=int(mid), vector=item_vecs[mid].tolist())
        for mid in range(n)
    ]
    for s in range(0, len(points), 500):
        client.upsert(collection_name=COLLECTION, points=points[s:s + 500])
    print(f"Uploaded {n:,} vectors to '{COLLECTION}'.")


def search(query_vec, k=5):
    if use_qdrant:
        hits = client.query_points(
            collection_name=COLLECTION,
            query=query_vec.tolist(),
            limit=k + 1,
        )
        return [(h.id, h.score) for h in hits.points]
    sims  = item_vecs @ query_vec
    order = np.argsort(-sims)[:k + 1]
    return [(int(j), float(sims[j])) for j in order]

C:\Users\shaur\AppData\Local\Temp\ipykernel_31196\522771714.py:2: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


Uploaded 1,683 vectors to 'collab_vectors'.


## 4 · Demo — ANN search

In [5]:
demo_id = 50 if 50 < n else 1    # movie 50 = Star Wars in MovieLens 100K

print(f"Movies nearest to '{title_of.get(demo_id)}' in collaborative taste space:")
for mid, score in search(item_vecs[demo_id], k=5):
    if mid != demo_id:
        print(f"  {score:.3f}  {title_of.get(mid)}")

print("\nThese neighbours share taste (who watches them together), not content.")
print("Compare with notebook 03 which groups by title/genre text.")
print("Next -> 07_evaluation.ipynb")

Movies nearest to 'Star Wars (1977)' in collaborative taste space:
  0.955  Return of the Jedi (1983)
  0.927  Godfather, The (1972)
  0.914  Fargo (1996)
  0.908  Toy Story (1995)
  0.854  Contact (1997)

These neighbours share taste (who watches them together), not content.
Compare with notebook 03 which groups by title/genre text.
Next -> 07_evaluation.ipynb
